In [1]:
from pathlib import Path

from model_ranking import (
    load_transfer_metric_results,
    match_model_names,
    to_target_transfer_correlations,
    correlation_table,
    avg_correlation_table,
    avg_correlation_to_latex,
    avg_correlation_to_latex_transposed,
    dataframe_to_latex_table_styled,
)

INFO: P [MainThread] 2025-11-17 12:53:23,651 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
performance_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/transfer_Finetuned_Def_performance_scores.json"
results = load_transfer_metric_results(performance_path)
performance_scores = results["performance_scores"]

Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/transfer_Finetuned_Def_performance_scores.json


In [3]:
transfer_metric_abbrevs = {
    "Transfer_Score": "TS",
    "LEEP": "LEEP",
    "NCTI": "NCTI",
    "LogME": "LogME",
    "Hscore": "Hscore",
    "GBC": "GBC",
    "Regularized_Hscore": "RegHscore",
    "Gaussian_LEEP": "NLEEP",
}

In [4]:
base_path = Path("/g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled/transfer_metric_results/direct_transfer_results2")


transfer_metric_names = []
transfer_metrics_dfs = []
for transfer_metric in transfer_metric_abbrevs.keys():
    path = base_path / f"mitochondria_{transfer_metric}.json"
    results = load_transfer_metric_results(path)
    targets = results['metadata']['targets']
    transfer_scores = results["transfer_scores"]
    transfer_scores = match_model_names(
        performance_scores=performance_scores,
        transfer_scores=transfer_scores,
    )
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=transfer_scores,
        performance_per_target=performance_scores,
    )
    correlation_df = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets)
    transfer_metric_abbrev = transfer_metric_abbrevs[transfer_metric]
    transfer_metric_names.append(transfer_metric_abbrev)
    transfer_metrics_dfs.append(correlation_df)

Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled/transfer_metric_results/direct_transfer_results2/mitochondria_Transfer_Score.json
Experiment: mitochondria_Transfer_Score
Targets: 4 (EPFL, Hmito, Rmito, VNC)
Source models: 15
Total transfers: 57
Info: No match found for transfer model 'E_model5' in target 'EPFL' - removing from result
Info: No match found for transfer model 'E_model_NA2' in target 'EPFL' - removing from result
Info: No match found for transfer model 'E_model_Res1' in target 'EPFL' - removing from result
Info: No match found for transfer model 'E_model_Unetr2' in target 'EPFL' - removing from result
Info: No match found for transfer model 'Hm_model4' in target 'Hmito' - removing from result
Info: No match found for transfer model 'Hm_model_NA2' in target 'Hmito' - removing from result
Info: No match found for transfer model 'Hm_model_Res1' in target 'Hmito' - removing from result
Info: No match f

### CCFV

In [6]:
base_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/ccfv/direct"

file_name = f"transfer_CCFV_scores.json"
ccfv_path = Path(base_path) / file_name
results = load_transfer_metric_results(ccfv_path)
transfer_scores = results["transfer_scores"]
targets = list(transfer_scores.keys())
transfer_scores = match_model_names(
        performance_scores=performance_scores,
        transfer_scores=transfer_scores,
    )
KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
    targets=targets,
    transfer_metric_per_target=transfer_scores,
    performance_per_target=performance_scores,
    )
df_mito_ccfv = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets)
print(df_mito_ccfv)

Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/ccfv/direct/transfer_CCFV_scores.json
Info: No match found for transfer model 'E_model_NA2' in target 'EPFL' - removing from result
Info: No match found for transfer model 'E_model_Res1' in target 'EPFL' - removing from result
Info: No match found for transfer model 'E_model5' in target 'EPFL' - removing from result
Info: No match found for transfer model 'E_model_Unetr2' in target 'EPFL' - removing from result
Info: No match found for transfer model 'Hm_model_NA2' in target 'Hmito' - removing from result
Info: No match found for transfer model 'Hm_model_Res1' in target 'Hmito' - removing from result
Info: No match found for transfer model 'Hm_model4' in target 'Hmito' - removing from result
Info: No match found for transfer model 'Hm_model_Unetr2' in target 'Hmito' - removing from result
Info: No match found for transfer model 'Rm_model_NA2' in target 'Rmito' - re

# Latex Table

In [7]:
dfs = [
    df_mito_ccfv,
] + transfer_metrics_dfs

metric_names = [
    "CCFV",
] + transfer_metric_names

latex_table = dataframe_to_latex_table_styled(
    dfs, 
    metric_names,
    caption="Transfer Metric Correlation scores for final-finetuned performance, Semantic Segmentation of Mitochondria with a range of Gaussian Input Perturbation Strengths. \textit{pval.} $<$ 0.05 (*), \textit{pval.} $<$ 0.01 (**).",
    label="tab:transfer_metric_mitochondria_gauss"
)
print(latex_table)

\begin{table}[tb]
\centering
\scriptsize
\caption{Transfer Metric Correlation scores for final-finetuned performance, Semantic Segmentation of Mitochondria with a range of Gaussian Input Perturbation Strengths. 	extit{pval.} $<$ 0.05 (*), 	extit{pval.} $<$ 0.01 (**).}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{@{}c c
   >{\columncolor{GreyTable}}c
   >{\columncolor{GreyTable}}c
   c c
   >{\columncolor{GreyTable}}c
   >{\columncolor{GreyTable}}c
   c c@{}}
\toprule
\multirow{2}{*}{Metric} & {} & \multicolumn{2}{c}{EPFL} & \multicolumn{2}{c}{Hmito} & \multicolumn{2}{c}{Rmito} & \multicolumn{2}{c}{VNC} \\
&  &  \cellcolor{SecondaryColumnColor} & \cellcolor{SecondaryColumnColor}\textit{pval.} &  & \textit{pval.} &  \cellcolor{SecondaryColumnColor} & \cellcolor{SecondaryColumnColor}\textit{pval.} &  & \textit{pval.} \\
\cmidrule{1-10}
\multirow{3}{*}{CCFV} & K$\tau$ & -0.16 & (0.57) & -0.42 & (0.10) & -0.53 & (*) & -0.27 & (0.25) \\
& S$\rho$ & -0.24 & (0.48) & -0.57 & (0.05) & -0.69 & (*

## Transposed Table

In [11]:
# Test the transposed format with the same data
latex_table_transposed = avg_correlation_to_latex_transposed(
    df_avg_list=avg_dfs,
    metric_name_list=metric_names,
    metric_spec_list=metric_specs,
)
print(latex_table_transposed)

\begin{table*}[htbp]
    \centering
    \setlength{\tabcolsep}{1.5pt}
    \begin{tabular}{cc|cc|cc|cc|cc|cc|cc|cc|cc|cc}
    \hline
    \multirow{2}{*}{Task} &  & \multicolumn{2}{c|}{CCFV} & \multicolumn{2}{c|}{TS} & \multicolumn{2}{c|}{LEEP} & \multicolumn{2}{c|}{NCTI} & \multicolumn{2}{c|}{LogME} & \multicolumn{2}{c|}{Hscore} & \multicolumn{2}{c|}{GBC} & \multicolumn{2}{c|}{RegHscore} & \multicolumn{2}{c}{NLEEP} \\
 &  & \textit{Avg.} & \textit{std.} & \textit{Avg.} & \textit{std.} & \textit{Avg.} & \textit{std.} & \textit{Avg.} & \textit{std.} & \textit{Avg.} & \textit{std.} & \textit{Avg.} & \textit{std.} & \textit{Avg.} & \textit{std.} & \textit{Avg.} & \textit{std.} & \textit{Avg.} & \textit{std.} \\
\hline
\multirow{3}{*}{\rotatebox[origin=c]{90}{Mito}} & K$\tau$ & -0.34 & ±0.2 & 0.28 & ±0.2 & 0.23 & ±0.3 & -0.02 & ±0.1 & 0.05 & ±0.2 & 0.21 & ±0.2 & 0.30 & ±0.1 & 0.21 & ±0.2 & 0.08 & ±0.2 \\
 & S$\rho$ & -0.47 & ±0.2 & 0.43 & ±0.3 & 0.36 & ±0.3 & -0.05 & ±0.2 & 0.14 & ±0.2 & 0.3